# Creative analytics quickstart

Phase 1 (brand profile) and Phase 2 (video features). Add `GEMINI_API_KEY` and `GITHUB_TOKEN` under Colab **Secrets** first. The repo is private, so cloning needs a GitHub token with read access to it.

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

In [ ]:
!git clone https://$GITHUB_TOKEN@github.com/atharva-google/creative-analytics.git
%cd creative-analytics
%pip install -q -e .

## Input
Edit the brand, product and video links.

In [ ]:
import json

jobs = [
    {
        "brand": "Nestle",
        "product": "KitKat",
        "video_links": [
            "https://www.youtube.com/watch?v=-WOLE1oSn5k",
        ],
    }
]
with open("input.json", "w") as f:
    json.dump(jobs, f, indent=2)

## Phase 1: brand profile
Review the printed profile. It defines the labels every video is tagged with. Edit the JSON file by hand if something is wrong or missing, then continue.

In [ ]:
!python -m creative_analytics profile --input input.json
!cat output/brand_profiles/*.json

## Phase 2: video features
Safe to re-run: videos that already have a row are skipped, failed ones are retried.

In [ ]:
!python -m creative_analytics extract --input input.json --max-workers 4

In [ ]:
import glob
import pandas as pd

rows = [json.loads(line) for path in glob.glob("output/video_features/*.jsonl")
        if not path.endswith((".errors.jsonl", ".custom_state.jsonl")) for line in open(path)]
pd.json_normalize(rows)

In [ ]:
print(json.dumps(rows[0], indent=2, ensure_ascii=False))